# Experiment: Naive Paraphrasers and False Positives
The goal of this experiment is to find out whether Naive Paraphrasers risk False Positives because they (hypothesis) know too much about the original text.
Hence, we hypothesize if the imposter model uses only Naive Paraphrasers (i.e. T5, Ollama, etc.) to generate imposters, it will likely lead to a high number of False Positives.
We therefore manually comment out the Non-Naive LLM-based imposter generators in the `LLMImposterGenerator` class (`ImposterGenerator.py`) and use only the Naive Paraphrasers.

## Dataset
We construct a dataset of original texts with candidates from a different author (i.e. their LLM generated paraphrase). The dataset should include:
- Original text
- Paraphrased text
- Label indicating which paraphraser generated the paraphrase

The original texts should be from a variety of sources to ensure diversity in genres.

Since we ensured that the Imposter Generators are only using Naive Paraphrasers, we can generate candidates via Non-Naive LLM-based Paraphrasers (e.g. TopicParaphraser, BulletPointParaphraser, etc.) to see if they lead to False Positives.

In [1]:
import os
from pathlib import Path
from genai_detection.config import CONFIG
from datasets import load_from_disk

# The construction of this cross-genre dataset is in file `genai_detection/dataset_util.py`.
dataset = load_from_disk(Path(os.getcwd()).resolve().parent / CONFIG.CROSS_GENRE)[
    "train"
].to_pandas()
print(f"Loaded Cross-genre dataset with {len(dataset)} examples.")
display(dataset)

Loaded Cross-genre dataset with 35 examples.


,category,disputed_text,candidate_text,same,pair,artificial_generation,authors
0,Blog,void in ten minutes my friends will arrive to ...,"malcolms eyes shuddered, and he felt weakness....",True,[void in ten minutes my friends will arrive to...,False,"[1171643, 1171643]"
1,Blog,forgive me for not updating the blog for a few...,"the many, many, many, many, many flip-flops of...",False,[forgive me for not updating the blog for a fe...,False,"[3721797, 1936076]"
2,Gutenberg,"now, what i want is, facts. teach these boys a...",i have endeavoured in this ghostly little book...,True,"[now, what i want is, facts. teach these boys ...",False,"[Charles Dickens, Charles Dickens]"
3,Gutenberg,the first ray of light which illumines the glo...,"letter to mrs. saville, england. st. petersbur...",False,[the first ray of light which illumines the gl...,False,"[Charles Dickens, (Godwin) Shelley]"
4,Student Essays,most of my personality is based on the strong ...,one day while working in the laboratory in the...,True,[most of my personality is based on the strong...,False,"[CRS933, CRS933]"
5,Student Essays,"this picture portrays two scientists, i would ...","this morning i woke up thinking, ""another long...",False,"[this picture portrays two scientists, i would...",False,"[NEB343, NGS252]"
6,Pan 20,feeling suddenly more alert she pushed herself...,"""okay,"" he held up his hands defensively, ""i k...",True,[feeling suddenly more alert she pushed hersel...,False,"[578300, 578300]"
7,Pan 20,"""i just wanted to know why zim is acting so we...","danielle chuckled. ""i hope she does brat, beca...",False,"[""i just wanted to know why zim is acting so w...",False,"[7587901, 492387]"
8,Blog,void in ten minutes my friends will arrive to ...,after a year of being in the company of my fri...,False,[void in ten minutes my friends will arrive to...,True,"[1171643, T5_ChatGPT]"
9,Blog,void in ten minutes my friends will arrive to ...,virgil's confession to his insane actions and ...,False,[void in ten minutes my friends will arrive to...,True,"[1171643, Ollama]"


## AV Models
Multiple AV models should be used to evaluate the paraphrased texts.

In [5]:
import os
from pathlib import Path
import numpy as np
from tqdm import tqdm
from genai_detection.detectors.impostor import ImpostorDetector
from genai_detection.detectors.unmasking import UnmaskingDetector
from genai_detection.detectors.ppmd import PPMdDetector

naive_imposter_detector = ImpostorDetector(
    path2imp=Path(os.getcwd()).resolve().parent / CONFIG.PATH2BLOG,
    imposter_technique="naive_llm",  # Use only naive paraphrasers for imposter generation
)
non_naive_imposter_detector = ImpostorDetector(
    path2imp=Path(os.getcwd()).resolve().parent / CONFIG.PATH2BLOG,
    imposter_technique="non_naive_llm",  # Use only non-naive paraphrasers for imposter generation
)
generalized_imposter_detector = ImpostorDetector(
    path2imp=Path(os.getcwd()).resolve().parent / CONFIG.PATH2BLOG,
    imposter_technique="llm",  # Use all paraphrasers for imposter generation
)
# unmasking_detector = UnmaskingDetector()
# ppmd_detector = PPMdDetector()


# for detector in [imposter_detector, unmasking_detector, ppmd_detector]:
for detector_name, detector in zip(
    ["Naive", "Non-Naive", "Generalized"],
    [
        naive_imposter_detector,
        non_naive_imposter_detector,
        generalized_imposter_detector,
    ],
):
    dataset[f"{detector_name}_score"] = np.nan
    for i in tqdm(
        dataset.index,
        desc=f"Processing {detector_name} scores",
    ):
        original_text = dataset.loc[i, "disputed_text"]
        paraphrased_text = dataset.loc[i, "candidate_text"]
        score = detector.get_score([original_text, paraphrased_text], normalize=False)
        dataset.loc[i, f"{detector_name}_score"] = np.round(score, 2)

Processing Naive scores:   0%|          | 0/35 [00:00<?, ?it/s]

Again, result:  {Transform the statement utilizing the same style and roughly equivalent quantity of phrases:

Original: "They never trained me for this."

Paraphrase: "I was not prepared for this situation."}
Again, result:  {Transforme la phrase dans un style similaire à l'original avec environ le même nombre de mots :

Réformulez la phrase dans un ton semblable à celui d'origine et avec approximativement le même nombre de mots :

Note: Gash of cloud vient d'un mur de nuages, hh. Détonnements se produisent près de l'aube de ce mur de nuages. Après un certain temps, ils s'estompent. Les habitants penseaient être des spectateurs et se rassemblèrent à l'extérieur pour une douce nuit d'feux d'artifice.

L'élévateur résonna avant d'arrêter brusquement. Le gardien s'effondra, rubriquet son front. Trop tard. Ils ont épuisé le courant. Les formations noires de kataphractes se sont rassemblées en une masse épaisse dans le ciel rougeoyant, un troupeau d'aigles se jetant sur et autour des plus 

Processing Naive scores:   0%|          | 0/35 [12:50<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
save_path = (
    Path(os.getcwd()).resolve().parent / "results" / "paraphrasing" / "experiments"
)
if not save_path.exists():
    save_path.mkdir(parents=True, exist_ok=True)
display(dataset.head())
dataset.to_csv(save_path / "exp_imp_gen_av_scores_cross_genre_dataset.csv", index=False)

,category,original_text,original_file_name,text_length,candidate_paraphraser,paraphrased_text,ImpostorDetector_score
0,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Ollama,the potential successor to the dalai lama has ...,100.0
1,News,"for much of the past century, the dalai lama h...",cnn_040725,582,BulletPoint,"in a striking development, the spiritual leade...",100.0
2,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Translation,"'pendant plus d'un siecle, le dalai-lama a ete...",100.0
3,Blog,this may be a long blog...got a lot of thought...,27,933,Ollama,the author's reflections on connections and de...,100.0
4,Blog,this may be a long blog...got a lot of thought...,27,933,BulletPoint,it's always fascinating for me to encounter co...,100.0


In [ ]:
# calculate rmse per paraphraser and AV model
import pandas as pd

# check for FPs
rmse_per_paraphraser = {}
paraphraser_names = [
    "T5_ChatGPT",
    "T5_Google_PAWS",
    "Ollama",
    "BulletPoint",
    "Task",
    "Topic",
    "Title",
    "Translation",
]
dataset["paraphraser"] = dataset["authors"].values[0][1]
fp_dataset = dataset[~dataset["same"]]
for paraphraser in dataset["paraphraser"].unique():
    if paraphraser not in paraphraser_names:
        continue  # Skip if the paraphrasers that are actually human authors
    rmse_per_paraphraser[paraphraser] = {}
    for detector_name, detector in zip(
        ["Naive", "Non-Naive", "Generalized"],
        [
            naive_imposter_detector,
            non_naive_imposter_detector,
            generalized_imposter_detector,
        ],
    ):
        scores = fp_dataset[fp_dataset["candidate_paraphraser"] == paraphraser][
            f"{detector_name}_score"
        ]
        rmse = np.sqrt(np.mean((scores - 0) ** 2))
        rmse_per_paraphraser[paraphraser][detector_name] = rmse
# to dataframe
rmse_df = pd.DataFrame(rmse_per_paraphraser).T
rmse_df.reset_index(inplace=True)
rmse_df.rename(columns={"index": "candidate_paraphraser"}, inplace=True)
# rmse_df.to_csv(save_path / "naive_paraphrase_rmse_per_paraphraser.csv", index=False)
print("RMSE per candidate_paraphraser among FPs:")
display(rmse_df)

RMSE per candidate_paraphraser:


,candidate_paraphraser,ImpostorDetector
0,Ollama,100.000000
1,BulletPoint,100.000000
2,Translation,95.924775


In [ ]:
# Check for FNs
fn_per_paraphraser = {}
fn_dataset = dataset[dataset["same"]]
for paraphraser in dataset["paraphraser"].unique():
    if paraphraser not in paraphraser_names:
        continue  # Skip if the paraphrasers that are actually human authors
    rmse_per_paraphraser[paraphraser] = {}
    for detector_name, detector in zip(
        ["Naive", "Non-Naive", "Generalized"],
        [
            naive_imposter_detector,
            non_naive_imposter_detector,
            generalized_imposter_detector,
        ],
    ):
        scores = fn_dataset[fn_dataset["candidate_paraphraser"] == paraphraser][
            f"{detector_name}_score"
        ]
        rmse = np.sqrt(np.mean((scores - 1) ** 2))
        rmse_per_paraphraser[paraphraser][detector_name] = rmse
# to dataframe
rmse_df = pd.DataFrame(rmse_per_paraphraser).T
rmse_df.reset_index(inplace=True)
rmse_df.rename(columns={"index": "candidate_paraphraser"}, inplace=True)
# rmse_df.to_csv(save_path / "naive_paraphrase_rmse_per_paraphraser.csv", index=False)
print("RMSE per candidate_paraphraser among FNs:")
display(rmse_df)